In [1]:
import numpy as np
import pandas as pd
import re
import ftfy
import html
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report

In [2]:
df_raw=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
def clean_text_light_title(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_light_article(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\b(a\s+href|href|img\s+src|nbsp|read\s+more|click\s+here)\b',
                  ' ', text, flags=re.IGNORECASE)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

import numpy as np
import pandas as pd
import re, html
import ftfy

df_raw = pd.read_csv("development.csv", delimiter=",", index_col="Id")

def _clean_basic(text, extra_blacklist=False):
    if pd.isna(text):
        return ""
    text = ftfy.fix_text(str(text))
    text = html.unescape(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    if extra_blacklist:
        text = re.sub(
            r"\b(a\s+href|href|img\s+src|nbsp|read\s+more|click\s+here)\b",
            " ",
            text,
            flags=re.IGNORECASE,
        )
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def build_features(df):
    df = df.copy()

    # source
    df["source"] = (
    df.get("source", "Other")
    .replace(["\\N", ""], np.nan)
    .fillna("Other")
    .astype(str)
    )

    # page_rank
    df["page_rank"] = pd.to_numeric(df.get("page_rank", 0.0), errors="coerce").fillna(0.0)

    # timestamp -> features (se manca, tutto NaT)
    dt = pd.to_datetime(df.get("timestamp", pd.Series(pd.NaT, index=df.index)), errors="coerce")
    df["has_date"] = dt.notna().astype(int)
    df["quarter"] = dt.dt.quarter.fillna(-1).astype(int)
    df["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # title/article -> text
    title = df.get("title", pd.Series("", index=df.index))
    article = df.get("article", pd.Series("", index=df.index)).replace("\\N", "").fillna("").astype(str)

    title_clean = title.apply(clean_text_light_title)
    article_clean = article.apply(clean_text_light_article)

    df["text"] = (title_clean + " " + title_clean + " [TITLE] " + article_clean).str.strip()

    return df[["text", "source", "page_rank", "has_date", "is_weekend", "quarter"]]

In [4]:
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

TARGET_COL = "label"
TEXT_COL = "text"
CAT_COLS = ["source"]
NUM_COLS = ["page_rank", "has_date", "is_weekend", "quarter"]

X = df_raw.drop(columns=[TARGET_COL])
y = df_raw[TARGET_COL]

X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)

text_pipe = Pipeline([
    ("vec", TfidfVectorizer(
        sublinear_tf=True,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        max_features=400_000,   # chi² poi seleziona
        lowercase=False
    )),
    ("chi2", SelectKBest(chi2, k=80_000))
])

preprocess = ColumnTransformer(
    transformers=[
        ("text", text_pipe, TEXT_COL),
        ("source", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=5,
            sparse_output=True
        ), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ]
)

pipe = Pipeline([
    ("features", FunctionTransformer(build_features, validate=False)),
    ("prep", preprocess),
    ("clf", LinearSVC(class_weight="balanced", loss="squared_hinge", random_state=42)),
])

# tuning "lambda": solo i due knob principali
param_dist = {
    "prep__text__chi2__k": [60_000, 80_000, 120_000, 160_000],
    "clf__C": np.logspace(-3, 0.5, 10),  # 0.001 -> ~3.16
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=12,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

y_val_pred = best_model.predict(X_val)

print("Best params:", search.best_params_)
print("Validation Macro F1:", f1_score(y_val, y_val_pred, average="macro"))

/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_94786/3826547794.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_94786/3826547794.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_94786/3826547794.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_94786/3826547794.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ens

Best params: {'prep__text__chi2__k': 80000, 'clf__C': 0.21544346900318845}
Validation Macro F1: 0.701854628984817


In [5]:
from sklearn.metrics import classification_report
import pandas as pd

print(classification_report(y_val, y_val_pred, digits=4))

report_df = (
    pd.DataFrame(classification_report(y_val, y_val_pred, output_dict=True, digits=4))
      .T
      .sort_values("f1-score", ascending=True)
)

print("\n=== 5 classi con F1 più basso ===")
print(report_df[["precision", "recall", "f1-score", "support"]].head(5))

              precision    recall  f1-score   support

           0     0.7397    0.7458    0.7427      4709
           1     0.7268    0.8041    0.7635      2118
           2     0.8209    0.8114    0.8161      2232
           3     0.6057    0.5083    0.5527      1995
           4     0.7777    0.9341    0.8487      1715
           5     0.5685    0.4592    0.5081      2611
           6     0.5860    0.8129    0.6811       620

    accuracy                         0.7091     16000
   macro avg     0.6893    0.7251    0.7019     16000
weighted avg     0.7028    0.7091    0.7027     16000


=== 5 classi con F1 più basso ===
              precision    recall  f1-score  support
5              0.568516  0.459211  0.508051   2611.0
3              0.605735  0.508271  0.552739   1995.0
6              0.586047  0.812903  0.681081    620.0
macro avg      0.689348  0.725106  0.701855  16000.0
weighted avg   0.702803  0.709063  0.702708  16000.0


In [8]:
# Fit finale su TUTTO development
best_model.fit(df_raw, y) #SOSTITURE df_raw con X

# Leggi evaluation GREZZO (index_col="Id" così l'Id è l'indice)
df_eval = pd.read_csv("evaluation.csv", delimiter=",", index_col="Id")

pred_eval = best_model.predict(df_eval)

# Output richiesto: colonna 'Id' e colonna 'Predicted'
out = pd.DataFrame({
    "Id": df_eval.index,
    "Predicted": pred_eval
})

out.to_csv("submission.csv", index=False)
print(out.head())

   Id  Predicted
0   0          5
1   1          2
2   2          5
3   3          0
4   4          0
